# PIIS: Causal Telemetry Anomaly Detection

This notebook explores the statistical and machine-learning anomaly detection algorithms used in PIIS.

### Key Principles
- **Strict Causal Aggregations:** Calculations use trailing rolling windows to prevent lookahead data leakage.
- **Multi-Model Evaluator:** Combines 3-Sigma Dynamic Thresholding and Isolation Forest algorithms.
- **Telecom Signal Focus:** Evaluates radio frequency VSWR, optical Rx power attenuation, and UPF packet drop ratios.


In [ ]:
import sys
import os
import numpy as np
from datetime import datetime, timezone, timedelta

sys.path.insert(0, os.path.abspath('../src'))

from piis.anomaly.detector import TelemetryAnomalyPipeline

print('Anomaly detection pipeline imported.')


## 1. Generating Baseline and Degraded Telemetry Streams
Simulate 60 minutes of nominal VSWR telemetry on a 5G gNodeB followed by an abrupt antenna feeder degradation.


In [ ]:
np.random.seed(42)
base_time = datetime.now(timezone.utc) - timedelta(minutes=60)

# 45 nominal minutes (mean = 1.15, std = 0.05)
nominal = np.random.normal(1.15, 0.05, 45)
# 15 degraded minutes (feeder degradation, mean = 1.95, std = 0.10)
degraded = np.random.normal(1.95, 0.10, 15)
vswr_values = np.concatenate([nominal, degraded])

timestamps = [(base_time + timedelta(minutes=i)).isoformat() for i in range(60)]
print(f'Generated {len(vswr_values)} time-series samples.')


## 2. Running Anomaly Evaluation
Feed the samples into the detector to observe dynamic threshold breach detection.


In [ ]:
detector = TelemetryAnomalyPipeline()
anomalies_detected = []

for ts, val in zip(timestamps, vswr_values):
    res = detector.evaluate_sample(
        metric_name='vswr_ratio',
        network_element_id='GNB-RAN-701',
        value=float(val),
        timestamp=ts
    )
    if res.is_anomaly:
        anomalies_detected.append((ts, val, res.severity, res.z_score))

print(f'Total Anomaly Windows Triggered: {len(anomalies_detected)}')
if anomalies_detected:
    print('First Detected Anomaly:')
    print(f'  Timestamp: {anomalies_detected[0][0]}')
    print(f'  Value: {anomalies_detected[0][1]:.3f}')
    print(f'  Severity: {anomalies_detected[0][2]}')
    print(f'  Z-Score: {anomalies_detected[0][3]:.2f}')
